# Marine Acoustic Monitoring — Colab (Perch)

Este notebook replica la lógica de `COLAB_MARINE_ACOUSTIC.ipynb`, pero usando **Perch** (vía `bioacoustics_model_zoo`) como extractor de características en lugar de BirdNET.

Flujo general:
1. Clonar el repo del hackathon.
2. Subir `participant-download.env` con credenciales R2.
3. Descargar el subset `marine-acoustic-colab`.
4. Segmentar audios en clips de 5 s.
5. Extraer incrustaciones con Perch.
6. Reducir a 2D con UMAP y agrupar con HDBSCAN.
7. Explorar clusters en un mapa interactivo y escuchar clips.


In [1]:
# 1. Clonar repo del hackathon y entrar al directorio

!git clone https://github.com/SALA-AI-LATAM/hackathon-participants.git
%cd hackathon-participants


Cloning into 'hackathon-participants'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 150 (delta 85), reused 113 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 579.59 KiB | 34.09 MiB/s, done.
Resolving deltas: 100% (85/85), done.
/content/hackathon-participants


In [2]:
# 2. Subir credenciales R2 (participant-download.env)

from google.colab import files
import os
from pathlib import Path

def _load_from_secrets():
    try:
        from google.colab import userdata
        keys = ['R2_ENDPOINT', 'R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY', 'R2_BUCKET']
        if all(userdata.get(k) for k in keys):
            for k in keys:
                os.environ[k] = userdata.get(k)
            print('Credenciales cargadas desde Colab Secrets.')
            return True
    except Exception:
        pass
    return False

if not _load_from_secrets():
    print('Abriendo dialogo de subida... sube participant-download.env')
    try:
        uploaded = files.upload()
        if uploaded is None:
            raise RuntimeError(
                'files.upload() devolvio None. '
                'Ejecuta esta celda sola (no con Run All) o usa Colab Secrets.'
            )
    except RuntimeError:
        raise
    except Exception as e:
        raise RuntimeError(
            f'No se pudo abrir el dialogo de subida: {e}. '
            'Solucion: ejecuta esta celda sola o configura Colab Secrets.'
        )

    env_name = 'participant-download.env'
    if env_name not in uploaded:
        for k in uploaded:
            if 'env' in k or k.endswith('.env'):
                env_name = k
                break

    if env_name in uploaded:
        Path(env_name).write_bytes(uploaded[env_name])
        for line in Path(env_name).read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#'):
                key, val = line.replace('export ', '').split('=', 1)
                os.environ[key] = val.strip("'\"")
        print(f'Credenciales cargadas desde {env_name}')
    else:
        raise RuntimeError('No se encontro participant-download.env entre los archivos subidos.')


Abriendo dialogo de subida... sube participant-download.env


Saving participant-download.env to participant-download.env
Credenciales cargadas desde participant-download.env


In [3]:
# 3. Descargar subset marine-acoustic-colab desde R2

import os
import tarfile
from pathlib import Path

!pip install -q boto3 tqdm

import r2_download as hd

REPO_DIR = Path("/content/hackathon-participants")
DEST_DIR = REPO_DIR / "hackathon_data"
DEST_DIR.mkdir(parents=True, exist_ok=True)

client = hd.get_s3_client()
manifest = hd.load_manifest(
    bucket=os.environ["R2_BUCKET"],
    s3_client=client,
    cache_path=str(REPO_DIR / "manifest.json"),
)
hd.summarize_manifest(manifest)

DATASET = "marine-acoustic-colab"
stats = hd.download_dataset(
    manifest,
    dataset_name=DATASET,
    dest_dir=str(DEST_DIR),
)
print(f"\nDownloaded: {stats['downloaded']}, Skipped: {stats['skipped']}, Failed: {stats['failed']}")

tar_path = DEST_DIR / "marine-acoustic-colab" / "marine-acoustic-colab.tar.gz"
if tar_path.exists():
    extract_to = DEST_DIR / "marine-acoustic"
    extract_to.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=extract_to)
    print(f"Extraído en {extract_to}")
    !ls -la "{extract_to}"
else:
    raise FileNotFoundError("No se encontró marine-acoustic-colab.tar.gz")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.3 MB/s eta 0:00:00
Dataset               Shards  Size (GB) Format     Description
--------------------------------------------------------------------------------
precipitation-nowcasting      14       2.38 csv+netcdf+pt Weather station CSVs + LDAS gr
marine-acoustic-colab       1       0.44 tar.gz     Colab-friendly subset of under
marine-acoustic-core     146       7.33 wav        Curated subset of underwater a
bruv-videos               18      69.48 mp4        BRUV underwater video files — 


Shards:   0%|          | 0/1 [00:00<?, ?shard/s, file=marine-acoustic-colab.tar.gz, size_mb=445]
marine-acoustic-colab.tar.gz:   0%|          | 0.00/445M [00:00<?, ?B/s]
marine-acoustic-colab.tar.gz:   0%|          | 262k/445M [00:00<18:04, 410kB/s]
marine-acoustic-colab.tar.gz:   1%|          | 2.88M/445M [00:00<01:26, 5.12MB/s]
marine-acoustic-colab.tar.gz:   2%|▏         | 8.65M/445M [00:01<00:48, 9.03MB/s]
marine-acoustic-colab.tar.gz:   4%|▍         | 17.0M/445M [00:01<00:25, 16.5MB/s]
marine-acoustic-colab.tar.gz:   7%|▋         | 30.4M/445M [00:01<00:05, 78.6MB/s]
marine-acoustic-colab.tar.gz:   8%|▊         | 34.1M/445M [00:01<00:04, 101MB/s] 
marine-acoustic-colab.tar.gz:   8%|▊         | 37.7M/445M [00:01<00:04, 101MB/s]
marine-acoustic-colab.tar.gz:  11%|█         | 48.2M/445M [00:01<00:03, 101MB/s]
marine-acoustic-colab.tar.gz:  13%|█▎        | 58.5M/445M [00:01<00:03, 105MB/s]
marine-acoustic-colab.tar.gz:  15%|█▌        | 67.1M/445M [00:01<00:03, 105MB/s]
marine-acoustic-

  Verifying checksum (445 MB)... 

Shards: 100%|██████████| 1/1 [00:05<00:00,  5.43s/shard, file=marine-acoustic-colab.tar.gz, size_mb=445]

ok

Downloaded: 1, Skipped: 0, Failed: 0



/tmp/ipykernel_2557/3203148242.py:36: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_to)


Extraído en /content/hackathon-participants/hackathon_data/marine-acoustic
total 12
drwxr-xr-x 3 root    root  4096 Mar 11 19:26 .
drwxr-xr-x 4 root    root  4096 Mar 11 19:26 ..
drwxr-xr-x 5 1000104 users 4096 Mar  9 19:26 marine-acoustic


In [6]:
# 4. Instalar dependencias (audio, Perch, UMAP/HDBSCAN, interfaz)

!pip install -q numpy soundfile scipy matplotlib boto3 tqdm librosa scikit-maad
!pip install -q opensoundscape==0.12.1 bioacoustics-model-zoo==0.12.0 tensorflow tensorflow-hub
!pip install -q umap-learn hdbscan pyarrow pandas plotly ipywidgets
!pip install -q ai-edge-litert


In [24]:
import numpy as np
import pandas as pd
from pathlib import Path
import soundfile as sf
import librosa
from scipy.signal import butter, sosfilt
import tensorflow as tf
import tensorflow_hub as hub
import umap
import hdbscan
import tempfile
import os

# --- Config ---
DATA_DIR = Path("/content/hackathon-participants/hackathon_data/marine-acoustic/marine-acoustic")
OUT_DIR = Path("/content/hackathon-participants/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_CLIPS = 500
CLIP_DURATION = 5.0
PERCH_SR = 32000

# --- 1. Descubrir WAVs ---
wavs = sorted(DATA_DIR.rglob("*.wav"))
print(f"Found {len(wavs)} WAV files")

# --- 2. Segmentar en clips ---
tmp_dir = Path(tempfile.mkdtemp())
clips_meta = []
clip_id = 0

for wav_path in wavs:
    try:
        info = sf.info(str(wav_path))
        sr_orig = info.samplerate
        total_s = info.duration
    except Exception as e:
        print(f"Skip {wav_path.name}: {e}")
        continue

    for start_s in np.arange(0, total_s - CLIP_DURATION, CLIP_DURATION):
        if clip_id >= MAX_CLIPS:
            break
        try:
            audio, sr = sf.read(
                str(wav_path),
                start=int(start_s * sr_orig),
                stop=int((start_s + CLIP_DURATION) * sr_orig),
                dtype="float32",
            )
            if audio.ndim > 1:
                audio = audio.mean(axis=1);
            sos = butter(4, 50, btype="highpass", fs=sr, output="sos")
            audio = sosfilt(sos, audio);
            if sr != PERCH_SR:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=PERCH_SR);
            clip_path = tmp_dir / f"clip_{clip_id:04d}.wav";
            sf.write(str(clip_path), audio, PERCH_SR);
            unit = "pilot";
            for u in ["5783", "6478"]:
                if u in str(wav_path):
                    unit = u;
                    break;
            clips_meta.append({
                "clip_id": clip_id,
                "file_path": str(wav_path),
                "clip_wav": str(clip_path),
                "start_s": float(start_s),
                "end_s": float(start_s + CLIP_DURATION),
                "unit": unit,
                "source_group": wav_path.parent.name,
            });
            clip_id += 1;
        except Exception as e:
            print(f"Error clip {clip_id}: {e}");
            continue;
    if clip_id >= MAX_CLIPS:
        break;

print(f"Prepared {len(clips_meta)} clips");

# --- 3. Cargar Perch desde TF Hub ---
PERCH_URL = "https://kaggle.com/models/google/bird-vocalization-classifier/TensorFlow2/bird-vocalization-classifier/4";
print("Loading Perch model from TF Hub...");
perch_model = hub.load(PERCH_URL);
print("Perch model loaded.");

clip_files = [c["clip_wav"] for c in clips_meta];
print(f"Generating Perch embeddings for {len(clip_files)} clips...");

embeddings = [];
for clip_path in clip_files:
    audio, _ = sf.read(clip_path, dtype="float32");
    if audio.ndim > 1:
        audio = audio.mean(axis=1);
    audio_tensor = tf.constant(audio[np.newaxis, :], dtype=tf.float32);
    outputs = perch_model.infer_tf(audio_tensor);
    # outputs es tupla: (logits, embeddings shape (1, frames, 1280))
    emb_tensor = outputs[1] if isinstance(outputs, tuple) else outputs["embedding"];
    emb = emb_tensor.numpy().mean(axis=1).flatten();
    embeddings.append(emb);

emb_array = np.array(embeddings).astype(np.float32);
print(f"Embeddings shape: {emb_array.shape}");

# --- 4. UMAP + HDBSCAN ---
print("Running UMAP + HDBSCAN...");
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42);
umap_2d = reducer.fit_transform(emb_array);

clusterer = hdbscan.HDBSCAN(min_cluster_size=5);
labels = clusterer.fit_predict(emb_array);

# --- 5. Guardar resultados ---
clips_df = pd.DataFrame(clips_meta).drop(columns=["clip_wav"]);
clips_df["cluster"] = labels[:len(clips_df)];

clips_df.to_parquet(OUT_DIR / "clips_with_clusters_perch.parquet", index=False);
np.save(OUT_DIR / "umap_2d_perch.npy", umap_2d[:len(clips_df)]);
np.save(OUT_DIR / "embeddings_perch.npy", emb_array);

print(f"Saved to {OUT_DIR}");
print(f"Clusters found: {len(set(labels) - {-1})}");
print(f"Noise points: {(labels == -1).sum()}");

import shutil
# shutil.rmtree(tmp_dir, ignore_errors=True) # Commented out to prevent deletion of temporary files

Found 11 WAV files
Prepared 500 clips
Loading Perch model from TF Hub...
Perch model loaded.
Generating Perch embeddings for 500 clips...
Embeddings shape: (500, 1)
Running UMAP + HDBSCAN...


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Saved to /content/hackathon-participants/outputs
Clusters found: 15
Noise points: 49


In [30]:
# 6. Interfaz interactiva: UMAP + audio (Perch)

import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
from IPython.display import Audio, display
import ipywidgets as widgets
import librosa

OUT_DIR = Path("/content/hackathon-participants/outputs")

clips_path = OUT_DIR / "clips_with_clusters_perch.parquet"
umap_path = OUT_DIR / "umap_2d_perch.npy"

if not clips_path.exists() or not umap_path.exists():
    raise FileNotFoundError(
        "No se encontraron resultados del pipeline Perch. Ejecuta primero la celda de embeddings + UMAP + HDBSCAN."
    )

clips_df = pd.read_parquet(clips_path)
umap_2d = np.load(umap_path)

# Alinear longitudes por seguridad
n = min(len(clips_df), len(umap_2d))
clips_df = clips_df.iloc[:n].copy()
clips_df["umap_x"] = umap_2d[:n, 0]
clips_df["umap_y"] = umap_2d[:n, 1]
clips_df["cluster_str"] = clips_df["cluster"].astype(str)

# Scatter interactivo Plotly
fig = px.scatter(
    clips_df,
    x="umap_x",
    y="umap_y",
    color="cluster_str",
    hover_data=["clip_id", "unit", "source_group", "start_s", "end_s", "file_path"],
    title="Marine Acoustic — UMAP + HDBSCAN (Perch embeddings)",
)
fig.show()

# Widgets para explorar clusters y escuchar clips
clusters = sorted(clips_df["cluster"].unique())

cluster_dropdown = widgets.Dropdown(
    options=clusters,
    description="Cluster:",
    value=clusters[0],
)

clip_dropdown = widgets.Dropdown(
    options=[],
    description="Clip:",
)

output_audio = widgets.Output()


def update_clips(*args):
    cluster_val = cluster_dropdown.value
    subset = clips_df[clips_df["cluster"] == cluster_val]
    subset = subset.head(50)
    options = [
        (
            f"id={row.clip_id} | {row.unit} | {row.source_group} | {row.start_s:.1f}-{row.end_s:.1f}s",
            int(row.clip_id),
        )
        for _, row in subset.iterrows()
    ]
    if options:
        clip_dropdown.options = options
        clip_dropdown.value = options[0][1]
    else:
        clip_dropdown.options = []


cluster_dropdown.observe(update_clips, names="value")
update_clips()


def play_selected_clip(b):
    output_audio.clear_output()
    if not clip_dropdown.options:
        return
    clip_id = clip_dropdown.value
    row = clips_df[clips_df["clip_id"] == clip_id].iloc[0]
    file_path = row["file_path"]
    start_s = float(row["start_s"])
    duration_s = float(row["end_s"] - row["start_s"])

    audio, sr = librosa.load(file_path, sr=None, offset=start_s, duration=duration_s)
    with output_audio:
        display(Audio(audio, rate=sr))


play_button = widgets.Button(description="Reproducir clip")
play_button.on_click(play_selected_clip)

widgets.VBox([
    widgets.HBox([cluster_dropdown, clip_dropdown, play_button]),
    output_audio,
])


In [25]:
import soundfile as sf
import tensorflow as tf
import numpy as np

test_audio, _ = sf.read(clip_files[0], dtype="float32")
if test_audio.ndim > 1:
    test_audio = test_audio.mean(axis=1)
test_tensor = tf.constant(test_audio[np.newaxis, :], dtype=tf.float32)
outputs = perch_model.infer_tf(test_tensor)

print(type(outputs))
print(len(outputs))
for i, o in enumerate(outputs):
    print(f"outputs[{i}] shape: {o.shape}")

<class 'tuple'>
2
outputs[0] shape: (1, 10932)
outputs[1] shape: (1, 1280)


In [26]:
  embeddings = []
  for clip_path in clip_files:
      audio, _ = sf.read(clip_path, dtype="float32")
      if audio.ndim > 1:
          audio = audio.mean(axis=1)
      audio_tensor = tf.constant(audio[np.newaxis, :], dtype=tf.float32)
      outputs = perch_model.infer_tf(audio_tensor)
      emb = outputs[1].numpy()[0]  # shape (1280,)
      embeddings.append(emb)

  emb_array = np.array(embeddings).astype(np.float32)
  print(f"Embeddings shape: {emb_array.shape}")

  reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
  umap_2d = reducer.fit_transform(emb_array)
  clusterer = hdbscan.HDBSCAN(min_cluster_size=5)
  labels = clusterer.fit_predict(emb_array)

  clips_df = pd.DataFrame(clips_meta).drop(columns=["clip_wav"])
  clips_df["cluster"] = labels[:len(clips_df)]
  clips_df.to_parquet(OUT_DIR / "clips_with_clusters_perch.parquet", index=False)
  np.save(OUT_DIR / "umap_2d_perch.npy", umap_2d[:len(clips_df)])
  np.save(OUT_DIR / "embeddings_perch.npy", emb_array)

  print(f"Clusters found: {len(set(labels) - {-1})}")
  print(f"Noise points: {(labels == -1).sum()}")

Embeddings shape: (500, 1280)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



Clusters found: 2
Noise points: 2


In [28]:
OUT_DIR = Path("/content/hackathon-participants/outputs")
!ls -lh "{OUT_DIR}"

total 2.5M
-rw-r--r-- 1 root root 9.9K Mar 11 19:59 clips_with_clusters_perch.parquet
-rw-r--r-- 1 root root 2.5M Mar 11 19:59 embeddings_perch.npy
-rw-r--r-- 1 root root 4.1K Mar 11 19:59 umap_2d_perch.npy


In [31]:
import os
for f in os.listdir(OUT_DIR):
    size = os.path.getsize(OUT_DIR / f)
    print(f"{size/1024/1024:.2f} MB  {f}")

0.00 MB  umap_2d_perch.npy
0.01 MB  clips_with_clusters_perch.parquet
2.44 MB  embeddings_perch.npy


In [33]:
import pandas as pd
clips_df = pd.read_parquet(OUT_DIR / "clips_with_clusters_perch.parquet")
print(clips_df.shape)
print(clips_df.dtypes)
clips_df.head()

(500, 7)
clip_id           int64
file_path        object
start_s         float64
end_s           float64
unit             object
source_group     object
cluster           int64
dtype: object


,clip_id,file_path,start_s,end_s,unit,source_group,cluster
0,0,/content/hackathon-participants/hackathon_data...,0.0,5.0,5783,5783,0
1,1,/content/hackathon-participants/hackathon_data...,5.0,10.0,5783,5783,0
2,2,/content/hackathon-participants/hackathon_data...,10.0,15.0,5783,5783,0
3,3,/content/hackathon-participants/hackathon_data...,15.0,20.0,5783,5783,0
4,4,/content/hackathon-participants/hackathon_data...,20.0,25.0,5783,5783,0


In [34]:
  print(clips_df["cluster"].value_counts().sort_index())

cluster
-1      2
 0     26
 1    472
Name: count, dtype: int64
